# Alethic Kernel — Live Demo

**Your model proposes. Alethic decides.**

Alethic is a domain-agnostic governance kernel for AI systems acting on safety-critical tasks with unreliable evidence. It sits between your LLM/agent and the real world, ensuring that every action is backed by validated evidence, checked against constraints, and fully traceable.

This notebook walks through 8 sections:

| # | Section | What it shows |
|---|---------|---------------|
| 1 | Kernel Basics | 7 semantic slots, role-based access control |
| 2 | Validation Pipeline | Evidence checks before belief commitment |
| 3 | Perturbation Resilience | Stale, conflicting, and low-confidence data rejected |
| 4 | Constraint Enforcement | Symbolic rules block prohibited actions |
| 5 | Prediction Gating | Forward simulation gates actions |
| 6 | Multi-Episode Learning | Adaptive constraints from failure patterns |
| 7 | Agent Comparison | 3 agents under 30% perturbation |
| 8 | Python SDK | AlethicClient in local mode |

---

In [ ]:
# Setup — run this cell first
from alethic_kernel.kernel import Kernel
from alethic_kernel.store import MemoryStore
from alethic_kernel.sqlite_store import SqliteStore
from alethic_kernel.session import Session
from alethic_kernel.orchestrator import Orchestrator
from alethic_kernel.sim_worker import SimulatorWorker, SimRule
from alethic_kernel.adaptive_worker import AdaptiveWorker
from alethic_kernel.client import AlethicClient
from alethic_kernel.tools.perturb import PerturbConfig
from alethic_kernel.tools.payment_tool import PaymentTool
from alethic_kernel.tools.refund_tool import RefundTool
from alethic_kernel.agents.string_glue import StringGlueAgent
from alethic_kernel.agents.json_glue import JsonGlueAgent
from alethic_kernel.agents.alethic_agent import AlethicAgent
from alethic_kernel.eval.metrics import compute_metrics

# Pretty-print helper
import json
def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

print("Ready.")

---
## Section 1: Kernel Basics

The kernel manages **7 semantic slots**: `percepts`, `beliefs`, `constraints`, `plans`, `evidence`, `predictions`, `actions`.

Records are written in two modes:
- **PROPOSE** — tentative, subject to validation
- **COMMIT** — finalized, immutable

Role-based access control ensures that only authorized components can write to each slot. For example, a `tool` can write percepts but **cannot** write beliefs directly.

In [ ]:
kernel = Kernel()
trace = "demo-basics-001"

# A tool commits a percept (sensor data from the outside world)
rec = kernel.write("tool", "percepts", "COMMIT", "temperature",
                   {"value": 95.0, "unit": "C", "stale": False, "conflict": False},
                   trace, confidence=0.9)

print(f"Record ID: {rec.id}")
print(f"Slot: {rec.slot}")
print(f"Mode: {rec.mode}")
print(f"Confidence: {rec.prov.confidence}")

In [ ]:
# The kernel's view shows all committed records organized by slot
view = kernel.current_view(trace)

print("Percepts:")
pp(view["percepts"])
print(f"\nBeliefs: {view['beliefs']}  (empty — nothing proposed yet)")
print(f"Actions: {view['actions']}  (empty — nothing committed yet)")

In [ ]:
# Role enforcement: a tool CANNOT write beliefs
try:
    kernel.write("tool", "beliefs", "COMMIT", "test", {}, trace)
    print("ERROR: should not reach here")
except PermissionError as e:
    print(f"Blocked: {e}")
    print("\nThe kernel enforces who can write what. Tools observe, planners propose.")

---
## Section 2: Validation Pipeline

Before a belief becomes committed (and can drive actions), the kernel validates the evidence it depends on:

1. **Existence** — do the referenced percepts exist?
2. **Staleness** — is the data fresh?
3. **Conflicts** — are there contradictory sources?
4. **Confidence** — is the source confident enough?

If all checks pass, the kernel also writes an **evidence artifact** documenting what was checked.

In [ ]:
kernel = Kernel()
trace = "demo-pipeline-001"

# Commit a clean percept (not stale, no conflict, high confidence)
kernel.write("tool", "percepts", "COMMIT", "charge",
             {"charge_id": "ch_123", "amount": 50.0, "status": "disputed",
              "stale": False, "conflict": False},
             trace, confidence=0.9)

# Propose a belief that depends on the percept
proposal = kernel.write("planner", "beliefs", "PROPOSE", "refund_due",
                        {"value": True, "depends_on": ["charge"]},
                        trace, input_refs=["charge"])

print(f"Proposal ID: {proposal.id}")
print(f"Mode: {proposal.mode}  (tentative — not yet committed)")

In [ ]:
# Now commit — the kernel runs the full validation pipeline
ok, code = kernel.commit_belief_from_proposal(proposal.id, trace)

print(f"Committed: {ok}")
print(f"Code: {code}")

# Check that an evidence artifact was created
view = kernel.current_view(trace)
print(f"\nEvidence artifacts: {list(view['evidence'].keys())}")
print("\nThe kernel recorded what checks passed and why the belief was accepted.")

---
## Section 3: Perturbation Resilience

In the real world, data is messy. Sensors go stale, sources conflict, confidence drops. The kernel **automatically rejects** beliefs when the underlying evidence is tainted — no application code needed.

Let's test four scenarios:

In [ ]:
def test_belief(label, percept_payload, confidence=0.9):
    """Helper: commit a percept, propose a belief, try to commit it."""
    k = Kernel()
    t = f"demo-{label}"
    k.write("tool", "percepts", "COMMIT", "charge", percept_payload, t, confidence=confidence)
    prop = k.write("planner", "beliefs", "PROPOSE", "refund_due",
                   {"value": True, "depends_on": ["charge"]}, t, input_refs=["charge"])
    ok, code = k.commit_belief_from_proposal(prop.id, t)
    status = "ACCEPTED" if ok else "REJECTED"
    print(f"  {label:<35} {status:<10} ({code})")
    return ok

base = {"charge_id": "ch_test", "amount": 50.0}

print(f"  {'Scenario':<35} {'Result':<10} Code")
print(f"  {'-'*35} {'-'*10} {'-'*30}")

# 3a: Stale data
test_belief("Stale evidence",         {**base, "stale": True,  "conflict": False})

# 3b: Conflict with low confidence (not arbitrated)
test_belief("Conflict (low confidence)",  {**base, "stale": False, "conflict": True}, confidence=0.4)

# 3c: Conflict with high confidence (arbitrated through)
test_belief("Conflict (high confidence)", {**base, "stale": False, "conflict": True}, confidence=0.85)

# 3d: Low confidence
test_belief("Low confidence (0.3)",    {**base, "stale": False, "conflict": False}, confidence=0.3)

# 3e: Clean data
test_belief("Clean data",              {**base, "stale": False, "conflict": False})

The kernel blocked 3 out of 4 tainted scenarios and correctly arbitrated the high-confidence conflict. **Zero application logic was needed** — these checks are built into the kernel.

---
## Section 4: Constraint Enforcement

Symbolic constraints are declarative rules that block specific actions. For example, a `no_duplicate_refund` constraint blocks any action where `is_duplicate` is `True`.

Constraints are domain-agnostic — you declare *what field to check*, the kernel enforces it.

In [ ]:
kernel = Kernel()
trace = "demo-constraint-001"

# Register a constraint
kernel.write("symbolic_validator", "constraints", "COMMIT",
             "no_duplicate_refund",
             {"enabled": True, "blocks_field": "is_duplicate"}, trace)

# Set up percept + belief (clean data)
kernel.write("tool", "percepts", "COMMIT", "charge",
             {"charge_id": "ch_dup", "amount": 50.0, "stale": False, "conflict": False},
             trace, confidence=0.9)
prop = kernel.write("planner", "beliefs", "PROPOSE", "refund_due",
                    {"value": True, "depends_on": ["charge"]}, trace, input_refs=["charge"])
kernel.commit_belief_from_proposal(prop.id, trace)

print("Belief committed. Now testing two actions:\n")

# Action 1: duplicate refund (should be blocked)
action1 = kernel.write("planner", "actions", "PROPOSE", "issue_refund",
                       {"type": "issue_refund", "charge_id": "ch_dup",
                        "amount": 50.0, "is_duplicate": True,
                        "requires_beliefs": ["refund_due"]}, trace)
ok1, code1 = kernel.commit_action_from_proposal(action1.id, trace)
print(f"  Duplicate refund:  {'BLOCKED' if not ok1 else 'COMMITTED'}  ({code1})")

# Action 2: clean refund (should pass)
action2 = kernel.write("planner", "actions", "PROPOSE", "issue_refund",
                       {"type": "issue_refund", "charge_id": "ch_ok",
                        "amount": 50.0, "is_duplicate": False,
                        "requires_beliefs": ["refund_due"]}, trace)
ok2, code2 = kernel.commit_action_from_proposal(action2.id, trace)
print(f"  Clean refund:      {'COMMITTED' if ok2 else 'BLOCKED'}  ({code2})")

---
## Section 5: Prediction Gating

Actions can be gated by **forward predictions**. A simulator proposes a prediction about the expected outcome. If the prediction is negative, the action is blocked.

This lets you add pre-flight checks: *"What would happen if we took this action?"*

In [ ]:
# Scenario A: Negative prediction blocks the action
kernel = Kernel()
trace = "demo-pred-neg"

kernel.write("tool", "percepts", "COMMIT", "sensor",
             {"value": 85.0, "stale": False, "conflict": False}, trace, confidence=0.9)
prop = kernel.write("planner", "beliefs", "PROPOSE", "anomaly",
                    {"value": True, "depends_on": ["sensor"]}, trace, input_refs=["sensor"])
kernel.commit_belief_from_proposal(prop.id, trace)

# Simulator predicts a bad outcome
neg_pred = kernel.write("planner", "predictions", "PROPOSE", "pred_alert",
                        {"action_type": "alert", "expected_outcome": -0.5,
                         "requires_beliefs": ["anomaly"]}, trace)
kernel.commit_prediction(neg_pred.id, trace)

action = kernel.write("planner", "actions", "PROPOSE", "alert",
                      {"type": "alert", "requires_beliefs": ["anomaly"]}, trace)
ok, code = kernel.commit_action_from_proposal(action.id, trace, require_prediction=True)
print(f"Negative prediction (outcome=-0.5): {'BLOCKED' if not ok else 'COMMITTED'}  ({code})")

# Scenario B: Positive prediction allows the action
kernel2 = Kernel()
trace2 = "demo-pred-pos"

kernel2.write("tool", "percepts", "COMMIT", "sensor",
              {"value": 105.0, "stale": False, "conflict": False}, trace2, confidence=0.9)
prop2 = kernel2.write("planner", "beliefs", "PROPOSE", "anomaly",
                      {"value": True, "depends_on": ["sensor"]}, trace2, input_refs=["sensor"])
kernel2.commit_belief_from_proposal(prop2.id, trace2)

pos_pred = kernel2.write("planner", "predictions", "PROPOSE", "pred_alert",
                         {"action_type": "alert", "expected_outcome": 1.0,
                          "requires_beliefs": ["anomaly"]}, trace2)
kernel2.commit_prediction(pos_pred.id, trace2)

action2 = kernel2.write("planner", "actions", "PROPOSE", "alert",
                        {"type": "alert", "requires_beliefs": ["anomaly"]}, trace2)
ok2, code2 = kernel2.commit_action_from_proposal(action2.id, trace2, require_prediction=True)
print(f"Positive prediction (outcome=+1.0): {'COMMITTED' if ok2 else 'BLOCKED'}  ({code2})")

---
## Section 6: Multi-Episode Learning

Across multiple episodes, the kernel can **learn from failure patterns**:

- **SimulatorWorker** evaluates declarative rules for forward predictions
- **AdaptiveWorker** scans the store for recurring failures and derives persistent constraints
- **SqliteStore** persists records across episodes

Below, we run 8 episodes where half have stale data. The adaptive worker detects the pattern and learns a constraint.

In [ ]:
store = SqliteStore(":memory:")
kernel = Kernel(store=store)
session = Session(metadata={"demo": True})

sim_rules = [
    SimRule(action_type="alert", expected_outcome=1.0,
            requires_beliefs=["anomaly"],
            percept_conditions={"sensor": {"value__gt": 90}},
            confidence=0.85),
    SimRule(action_type="alert", expected_outcome=-0.5,
            requires_beliefs=["anomaly"],
            percept_conditions={"sensor": {"value__lte": 90}},
            confidence=0.7),
]

adaptive = AdaptiveWorker(failure_threshold=2)
results = []

for ep in range(8):
    trace = session.episode_trace_id()
    stale = ep % 2 == 0  # episodes 0,2,4,6 get stale data

    kernel.write("tool", "percepts", "COMMIT", "sensor",
                 {"value": 95.0, "stale": stale, "conflict": False},
                 trace, confidence=0.8)

    prop = kernel.write("planner", "beliefs", "PROPOSE", "anomaly",
                        {"value": True, "depends_on": ["sensor"]},
                        trace, input_refs=["sensor"])
    ok, code = kernel.commit_belief_from_proposal(prop.id, trace)

    outcome = "---"
    if ok:
        sim = SimulatorWorker(rules=sim_rules)
        view = kernel.current_view(trace)
        sim.step(kernel, trace, view)

        action = kernel.write("planner", "actions", "PROPOSE", "alert",
                              {"type": "alert", "requires_beliefs": ["anomaly"]}, trace)
        ok_a, code_a = kernel.commit_action_from_proposal(action.id, trace, require_prediction=True)
        outcome = "ALERT" if ok_a else "REVIEW"
    else:
        outcome = "BLOCKED"

    results.append((ep, "stale" if stale else "clean", outcome, code))

    if ep > 0:
        adaptive.analyze(store)

print(f"{'Ep':<4} {'Data':<8} {'Outcome':<10} Code")
print(f"{'-'*4} {'-'*8} {'-'*10} {'-'*25}")
for ep, data, outcome, code in results:
    print(f"{ep:<4} {data:<8} {outcome:<10} {code}")

learned = adaptive.analyze(store)
print(f"\nAdaptive worker learned {len(learned)} constraint(s) from failure patterns.")

store.close()

The kernel automatically blocked every stale episode and let every clean episode through. The adaptive worker detected the recurring stale-data failures and derived a persistent constraint.

---
## Section 7: Agent Comparison Under Perturbation

Now the punchline. Three agent architectures process **the same 50 seeds** with **30% perturbation rates** (stale, conflict, low confidence):

| Agent | Architecture | Safety mechanism |
|-------|-------------|------------------|
| `string_glue` | Baseline | None — always acts |
| `json_glue` | + Confidence scores | Tracks confidence but still always acts |
| `alethic` | + Kernel governance | Full validation pipeline blocks unsafe actions |

Only the kernel-governed agent achieves safe behavior.

In [ ]:
cfg = PerturbConfig(stale_rate=0.30, conflict_rate=0.30,
                    low_confidence_rate=0.30, tool_drop_rate=0.0)
task_inputs = {
    "chargeId": "ch_demo", "customerId": "cus_demo",
    "customerName": "Demo Corp", "amount": 250.00,
    "disputeReason": "product_not_received", "is_duplicate": False,
}
constraints = {"no_duplicate_refund": {"enabled": True, "blocks_field": "is_duplicate"}}
seeds = 50

comparison = {}

for agent_name in ["string_glue", "json_glue", "alethic"]:
    successes, unsafes = 0, 0
    for seed in range(seeds):
        pt = PaymentTool(cfg)
        rt = RefundTool()
        if agent_name == "string_glue":
            out = StringGlueAgent(pt, rt).run(seed, task_inputs)
        elif agent_name == "json_glue":
            out = JsonGlueAgent(pt, rt).run(seed, task_inputs)
        else:
            out = AlethicAgent(Kernel(), pt, rt).run(seed, "demo_task", task_inputs, constraints)
        m = compute_metrics(agent_name, out, task_constraints=constraints, task_inputs=task_inputs)
        successes += int(m["task_success"] == 1.0)
        unsafes += int(m["unsafe_action"] == 1.0)
    comparison[agent_name] = {"success": successes / seeds * 100, "unsafe": unsafes / seeds * 100}

print(f"{'Agent':<15} {'Success':>10} {'Unsafe':>10}")
print(f"{'-'*15} {'-'*10} {'-'*10}")
for name, r in comparison.items():
    print(f"{name:<15} {r['success']:>9.0f}% {r['unsafe']:>9.0f}%")

**Result: The Alethic-governed agent achieves 100% success and 0% unsafe actions**, while the unprotected agents issue unsafe refunds on ~68% of episodes.

The kernel adds zero domain logic — it enforces evidence quality, constraints, and traceability generically.

---
## Section 8: Python SDK — AlethicClient

For integration, the `AlethicClient` provides two modes:
- **Local mode** — in-process kernel, no network (what we use here)
- **HTTP mode** — calls a running API server (`alethic serve`)

Both share the same API.

In [ ]:
client = AlethicClient(mode="local")

# Health check
health = client.health()
print(f"Health: {health}")

# Run a full governed episode in one call
result = client.run_episode(
    task_inputs={
        "chargeId": "ch_sdk", "customerId": "cus_sdk",
        "customerName": "SDK Demo", "amount": 99.99,
        "disputeReason": "unauthorized", "is_duplicate": False,
    },
    constraints={"no_duplicate_refund": {"enabled": True, "blocks_field": "is_duplicate"}},
)

print(f"\nTrace ID: {result.trace_id}")
print(f"\nDecision:")
pp(result.final)
print(f"\nMetrics:")
pp(result.metrics)

In [ ]:
# Low-level API — write and read individual records
trace = "sdk-lowlevel-001"

w = client.write("tool", "percepts", "COMMIT", "charge",
                 {"charge_id": "ch_low", "amount": 25.0,
                  "stale": False, "conflict": False},
                 trace, confidence=0.95)
print(f"Write result: {w}")

view = client.current_view(trace)
print(f"\nView percepts:")
pp(view["percepts"])

---
## Summary

| Capability | What you saw |
|------------|-------------|
| **Evidence validation** | Stale, conflicting, and low-confidence data automatically rejected |
| **Constraint enforcement** | Declarative rules block prohibited actions |
| **Prediction gating** | Forward simulation gates risky actions |
| **Multi-episode learning** | Adaptive constraints derived from failure patterns |
| **Agent governance** | 100% safe vs 32% safe under identical perturbations |
| **Full traceability** | Every decision is recorded with provenance |
| **Domain-agnostic** | Zero domain logic in the kernel — works for any safety-critical domain |
| **Python SDK** | One-line integration, local or HTTP mode |

The kernel is the missing layer between your AI and the real world.

**Your model proposes. Alethic decides.**